# 115. Voronoiセントロイドのエクスポート

## 目的
- NB111-114で有効性を確認したVoronoiセントロイドを永続化
- NumPy (.npy) とJSON (.json) の2形式で保存
- 実運用（Zope KeywordIndex / Firestore array-contains-any）で利用可能にする

## 保存対象
- Wikipedia 10K E5-base (EN+JA混合, 20K件) で学習したセントロイド
- C=256, 正規化済み (768次元)
- NB112で汎化性能を確認済み（ドメイン外データでも有効）

## 推奨構成（NB114の結論）
| 目標 | assign | P | EN R@10 | JA R@10 |
|------|--------|---|---------|---------|
| R@10≥90% | 3 | 4 | 91.1% | 95.3% |
| R@10≥85% | 2 | 4 | 86.4% | 86.9% |
| R@10≥80% | 1 | 5 | 81.0% | 82.8% |

In [1]:
import numpy as np
import json
from pathlib import Path
from sklearn.cluster import MiniBatchKMeans

DATA_DIR = Path('../data')
np.random.seed(42)

## 1. Wikipedia E5-base混合データでk-means学習

In [2]:
# Wikipedia E5-base embeddings
wiki_en = np.load(DATA_DIR / '10k_e5_base_en_embeddings.npy')
wiki_ja = np.load(DATA_DIR / '10k_e5_base_ja_embeddings.npy')
print(f'Wikipedia EN: {wiki_en.shape}')
print(f'Wikipedia JA: {wiki_ja.shape}')

# EN+JA混合（NB112, 114と同条件）
wiki_mixed = np.vstack([wiki_en, wiki_ja])
print(f'Wikipedia Mixed: {wiki_mixed.shape}')

# 正規化
norms = np.linalg.norm(wiki_mixed, axis=1, keepdims=True)
wiki_normed = wiki_mixed / norms
print(f'Normalized: norm mean={np.linalg.norm(wiki_normed, axis=1).mean():.6f}')

Wikipedia EN: (10000, 768)
Wikipedia JA: (9990, 768)
Wikipedia Mixed: (19990, 768)
Normalized: norm mean=1.000000


In [3]:
# k-means学習（C=256, NB111-114と同一パラメータ）
N_CLUSTERS = 256

kmeans = MiniBatchKMeans(
    n_clusters=N_CLUSTERS,
    random_state=42,
    batch_size=2048,
    n_init=3
)
labels = kmeans.fit_predict(wiki_normed)

# セントロイドを正規化
centroids = kmeans.cluster_centers_
centroids_normed = centroids / np.linalg.norm(centroids, axis=1, keepdims=True)

print(f'Centroids: {centroids_normed.shape}')
print(f'Centroid norms: mean={np.linalg.norm(centroids_normed, axis=1).mean():.6f}')

# パーティションサイズ統計
sizes = [np.sum(labels == c) for c in range(N_CLUSTERS)]
print(f'\nPartition sizes:')
print(f'  mean={np.mean(sizes):.1f}, std={np.std(sizes):.1f}')
print(f'  min={np.min(sizes)}, max={np.max(sizes)}')
print(f'  CV={np.std(sizes)/np.mean(sizes):.3f}')

Centroids: (256, 768)
Centroid norms: mean=1.000000

Partition sizes:
  mean=78.1, std=64.0
  min=1, max=303
  CV=0.819


## 2. NumPy形式で保存

In [4]:
npy_path = DATA_DIR / 'voronoi_centroids_256_e5base_mixed.npy'
np.save(npy_path, centroids_normed)

# 検証
loaded = np.load(npy_path)
assert np.allclose(loaded, centroids_normed)
print(f'Saved: {npy_path}')
print(f'  Shape: {loaded.shape}, dtype: {loaded.dtype}')
print(f'  File size: {npy_path.stat().st_size / 1024:.1f} KB')

Saved: ../data/voronoi_centroids_256_e5base_mixed.npy
  Shape: (256, 768), dtype: float32
  File size: 768.1 KB


## 3. JSON形式で保存

Webアプリやクライアント側で利用しやすい形式。メタデータも含める。

In [5]:
json_path = DATA_DIR / 'voronoi_centroids_256_e5base_mixed.json'

export_data = {
    'metadata': {
        'model': 'intfloat/multilingual-e5-base',
        'dimension': int(centroids_normed.shape[1]),
        'n_clusters': int(centroids_normed.shape[0]),
        'training_data': 'Wikipedia 10K EN + 10K JA (mixed)',
        'training_samples': int(len(wiki_mixed)),
        'normalized': True,
        'kmeans_params': {
            'random_state': 42,
            'batch_size': 2048,
            'n_init': 3,
        },
        'recommended_configs': {
            'high_recall': {'assign': 3, 'probes': 4, 'note': 'R@10>=90%'},
            'balanced': {'assign': 2, 'probes': 4, 'note': 'R@10>=85%'},
            'low_cost': {'assign': 1, 'probes': 5, 'note': 'R@10>=80%'},
        },
        'usage': {
            'zope': 'KeywordIndex pivot_ids, query with operator="or"',
            'firestore': 'array field pivot_ids, query with array-contains-any',
        },
    },
    'centroids': centroids_normed.tolist(),
}

with open(json_path, 'w') as f:
    json.dump(export_data, f, ensure_ascii=False)

# 検証
with open(json_path, 'r') as f:
    loaded_json = json.load(f)

loaded_centroids = np.array(loaded_json['centroids'])
assert loaded_centroids.shape == centroids_normed.shape
# float64 → JSON → float64 の精度確認
max_diff = np.max(np.abs(loaded_centroids - centroids_normed))

print(f'Saved: {json_path}')
print(f'  File size: {json_path.stat().st_size / 1024:.1f} KB')
print(f'  Centroids shape: {loaded_centroids.shape}')
print(f'  Max diff (roundtrip): {max_diff:.2e}')
print(f'\nMetadata:')
for k, v in loaded_json['metadata'].items():
    print(f'  {k}: {v}')

Saved: ../data/voronoi_centroids_256_e5base_mixed.json
  File size: 4234.3 KB
  Centroids shape: (256, 768)
  Max diff (roundtrip): 0.00e+00

Metadata:
  model: intfloat/multilingual-e5-base
  dimension: 768
  n_clusters: 256
  training_data: Wikipedia 10K EN + 10K JA (mixed)
  training_samples: 19990
  normalized: True
  kmeans_params: {'random_state': 42, 'batch_size': 2048, 'n_init': 3}
  recommended_configs: {'high_recall': {'assign': 3, 'probes': 4, 'note': 'R@10>=90%'}, 'balanced': {'assign': 2, 'probes': 4, 'note': 'R@10>=85%'}, 'low_cost': {'assign': 1, 'probes': 5, 'note': 'R@10>=80%'}}
  usage: {'zope': 'KeywordIndex pivot_ids, query with operator="or"', 'firestore': 'array field pivot_ids, query with array-contains-any'}


## 4. 利用例: ドキュメントへのpivot_ids割り当て

In [6]:
def assign_pivot_ids(embedding, centroids, n_assign=2):
    """ドキュメントのembeddingに対してpivot_idsを割り当てる
    
    Args:
        embedding: (D,) 正規化済みembedding
        centroids: (C, D) 正規化済みセントロイド
        n_assign: 割り当てるセントロイド数
    
    Returns:
        list[int]: pivot_ids (n_assign個)
    """
    sims = centroids @ embedding
    return np.argsort(-sims)[:n_assign].tolist()


def search_voronoi(query_embedding, centroids, n_probes=4):
    """クエリに対してprobe対象のpivot_idsを返す
    
    Args:
        query_embedding: (D,) 正規化済みクエリembedding
        centroids: (C, D) 正規化済みセントロイド
        n_probes: 探索するセントロイド数
    
    Returns:
        list[int]: probe対象のpivot_ids (n_probes個)
    """
    sims = centroids @ query_embedding
    return np.argsort(-sims)[:n_probes].tolist()


# デモ: Wikipedia ENの先頭5件に対してpivot_idsを割り当て
print('=== ドキュメントへのpivot_ids割り当て例 (assign=2) ===')
for i in range(5):
    emb = wiki_normed[i]
    pids = assign_pivot_ids(emb, centroids_normed, n_assign=2)
    print(f'  Doc {i}: pivot_ids = {pids}')

# デモ: クエリ検索
print('\n=== クエリ検索例 (P=4) ===')
query = wiki_normed[0]  # 先頭をクエリとして使用
probe_ids = search_voronoi(query, centroids_normed, n_probes=4)
print(f'  Query → probe pivot_ids = {probe_ids}')
print(f'  Zope:      catalog.searchResults(pivot_ids={{\"query\": {probe_ids}, \"operator\": \"or\"}})')
print(f'  Firestore: .where("pivot_ids", "array-contains-any", {probe_ids})')

=== ドキュメントへのpivot_ids割り当て例 (assign=2) ===
  Doc 0: pivot_ids = [68, 128]
  Doc 1: pivot_ids = [158, 221]
  Doc 2: pivot_ids = [171, 68]
  Doc 3: pivot_ids = [103, 251]
  Doc 4: pivot_ids = [188, 30]

=== クエリ検索例 (P=4) ===
  Query → probe pivot_ids = [68, 128, 56, 13]
  Zope:      catalog.searchResults(pivot_ids={"query": [68, 128, 56, 13], "operator": "or"})
  Firestore: .where("pivot_ids", "array-contains-any", [68, 128, 56, 13])


## 5. 保存ファイル一覧

In [7]:
print('='*60)
print('保存ファイル一覧')
print('='*60)

files = [
    ('NumPy', npy_path),
    ('JSON', json_path),
]

for fmt, path in files:
    size_kb = path.stat().st_size / 1024
    print(f'\n  [{fmt}] {path.name}')
    print(f'    Path: {path}')
    print(f'    Size: {size_kb:.1f} KB')

print(f'''
使い方:
  # NumPy
  centroids = np.load("data/voronoi_centroids_256_e5base_mixed.npy")

  # JSON (メタデータ付き)
  with open("data/voronoi_centroids_256_e5base_mixed.json") as f:
      data = json.load(f)
  centroids = np.array(data["centroids"])
  config = data["metadata"]["recommended_configs"]
''')

保存ファイル一覧

  [NumPy] voronoi_centroids_256_e5base_mixed.npy
    Path: ../data/voronoi_centroids_256_e5base_mixed.npy
    Size: 768.1 KB

  [JSON] voronoi_centroids_256_e5base_mixed.json
    Path: ../data/voronoi_centroids_256_e5base_mixed.json
    Size: 4234.3 KB

使い方:
  # NumPy
  centroids = np.load("data/voronoi_centroids_256_e5base_mixed.npy")

  # JSON (メタデータ付き)
  with open("data/voronoi_centroids_256_e5base_mixed.json") as f:
      data = json.load(f)
  centroids = np.array(data["centroids"])
  config = data["metadata"]["recommended_configs"]

